## Install necessary modules

In [5]:
pip install --upgrade "sagemaker>2.200.0" transformers datasets "accelerate>=0.28.0" "peft>=0.9.0" bitsandbytes "scikit-learn>=1.3.2" --quiet

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Note: you may need to restart the kernel to use updated packages.


## Import Libraries & Define Globals

In [ ]:
import sagemaker
import boto3
import os
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.workflow.model_step import ModelStep
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.huggingface import HuggingFace, HuggingFaceModel
from sagemaker.workflow.parameters import ParameterString

# --- SageMaker Session and Role Setup ---
sess = sagemaker.Session()
# sagemaker_session = sagemaker.Session()
bucket = sess.default_bucket()  # Or specify your own S3 bucket
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

print(f"SageMaker Role ARN: {role}")
print(f"SageMaker Session region: {region}")
print(f"S3 bucket: {bucket}")

# --- Pipeline Configuration ---
# Use a cache to speed up pipeline execution during development
cache_config = CacheConfig(enable_caching=True, expire_after="30d")

# Define a name for the model package group in the registry
model_package_group_name = "HuggingFace-Finetune-OPT-Pipeline"

## Create Preprocessor

In [2]:
%%writefile preprocess.py

import argparse
import os
import logging
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

# Set up logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def preprocess_data(data, tokenizer):
    """Tokenize the dialogue and summary."""
    inputs = ["summarize: " + dialogue for dialogue in data["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")
    
    # Setup the tokenizer for targets
    labels = tokenizer(text_target=data["summary"], max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_id", type=str, default="facebook/opt-125m")
    parser.add_argument("--dataset_name", type=str, default="samsum")
    args, _ = parser.parse_known_args()

    logger.info("Starting data preprocessing...")

    # Define output paths
    train_output_path = "/opt/ml/processing/train"
    test_output_path = "/opt/ml/processing/test"
    
    os.makedirs(train_output_path, exist_ok=True)
    os.makedirs(test_output_path, exist_ok=True)

    # Load dataset from Hugging Face Hub
    dataset = load_dataset(args.dataset_name)
    logger.info(f"Dataset loaded: {dataset}")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(args.model_id)
    
    # Preprocess and save datasets
    train_dataset = dataset["train"].map(lambda data: preprocess_data(data, tokenizer), batched=True)
    test_dataset = dataset["test"].map(lambda data: preprocess_data(data, tokenizer), batched=True)

    logger.info("Saving processed datasets to disk...")
    train_dataset.save_to_disk(train_output_path)
    test_dataset.save_to_disk(test_output_path)
    
    logger.info("Preprocessing complete.")

Overwriting preprocess.py


## Create Training Script

In [6]:
%%writefile train.py

import argparse
import os
import logging
from transformers import Trainer, TrainingArguments
from smexperiments.tracker import Tracker

# (Import other necessary libraries like torch, datasets, etc.)

def main():
    parser = argparse.ArgumentParser()
    # Your usual script arguments
    parser.add_argument("--epochs", type=int, default=1)
    parser.add_argument("--learning_rate", type=float, default=3e-4)
    parser.add_argument("--model_id", type=str)
    # SageMaker environment arguments
    parser.add_argument("--model_dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    
    args, _ = parser.parse_known_args()

    # --- SageMaker Experiments Integration ---
    # 1. Load the tracker from the file path passed by SageMaker
    tracker = Tracker.load()

    # 2. Log hyperparameters
    tracker.log_parameters({
        "learning_rate": args.learning_rate,
        "epochs": args.epochs,
        "model_id": args.model_id
    })
    
    # --- Load your dataset and model here ---
    # ... (your model and data loading code) ...

    # --- Define a custom callback for the Trainer to log metrics ---
    from transformers.trainer_callback import TrainerCallback

    class ExperimentTrackerCallback(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if state.is_world_process_zero:  # Only log on the main process
                for key, value in logs.items():
                    if isinstance(value, (int, float)):
                        # 3. Log metrics during training
                        tracker.log_metric(key, value, step=state.global_step)

    # --- Set up Trainer ---
    training_args = TrainingArguments(
        output_dir=args.model_dir,
        num_train_epochs=args.epochs,
        learning_rate=args.learning_rate,
        # ... other training arguments
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        callbacks=[ExperimentTrackerCallback()], # Add the callback here
    )

    # --- Start Training ---
    trainer.train()

    # --- Save the model ---
    trainer.save_model(args.model_dir)

    # 4. (Optional) Log output artifacts like a confusion matrix or config file
    # tracker.log_output_file("path/to/your/file.txt", "output_files")

    # 5. Close the tracker
    tracker.close()

if __name__ == "__main__":
    main()

Overwriting train.py


## Define the SageMaker Pipeline Steps

## Preprocessing step

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor

# Define the processor
sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="hf-llm-data-prep",
    role=role,
)

# Define the pipeline step
step_process = ProcessingStep(
    name="PreprocessHuggingFaceData",
    processor=sklearn_processor,
    inputs=[],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
    ],
    code="preprocess.py",
    cache_config=cache_config,
)

print("Processing step defined.")

## Training step

In [ ]:
# Define hyperparameters that will be passed to the training script
hyperparameters = {
    "epochs": 1,
    "per_device_train_batch_size": 8,
    "learning_rate": 3e-4,
    "model_id": "facebook/opt-125m",
}

# Define the HuggingFace Estimator
huggingface_estimator = HuggingFace(
    entry_point="train.py",
    instance_type="ml.g5.2xlarge", # A good GPU instance for this model
    instance_count=1,
    role=role,
    transformers_version="4.37",
    pytorch_version="2.1",
    py_version="py310",
    hyperparameters=hyperparameters,
    base_job_name="hf-llm-finetune",
)

# Define the pipeline step
step_train = TrainingStep(
    name="FineTuneHuggingFaceModel",
    estimator=huggingface_estimator,
    inputs={
        "train": sagemaker.inputs.TrainingInput(
            # The S3 URI is dynamically pulled from the output of the previous processing step
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="application/x-sagemaker-channel"
        )
    },
    cache_config=cache_config,
)

print("Training step defined.")

## Model Registration Step

In [ ]:
# Create a HuggingFaceModel object from the trained model artifacts
# This object defines how the model will be served for inference
model = HuggingFaceModel(
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    transformers_version="4.37",
    pytorch_version="2.1",
    py_version="py310",
)

# Define the ModelStep for registration
step_register_model = ModelStep(
   name="RegisterHuggingFaceModel",
   step_args=model.register(
      content_types=["application/json"],
      response_types=["application/json"],
      inference_instances=["ml.g5.2xlarge", "ml.t2.medium"],
      transform_instances=["ml.m5.xlarge"],
      model_package_group_name=model_package_group_name,
      approval_status="PendingManualApproval" # Can be set to "Approved" for auto-deployment
   )
)

print("Model registration step defined.")

## Create sagemaker experiments and a trial

In [ ]:
import sagemaker
from smexperiments.experiment import Experiment
from smexperiments.trial import Trial
from botocore.exceptions import ClientError
import time

# SageMaker session
sagemaker_session = sagemaker.Session()

# Create a unique name for your experiment
experiment_name = "llm-finetuning-experiment"

# Create an Experiment
try:
    sagemaker_experiment = Experiment.create(
        experiment_name=experiment_name,
        description="Fine-tuning a Hugging Face LLM.",
        sagemaker_boto_client=sagemaker_session.boto_session.client('sagemaker')
    )
    print(f"Created new experiment: {experiment_name}")
except ClientError:
    sagemaker_experiment = Experiment.load(
        experiment_name=experiment_name,
        sagemaker_boto_client=sagemaker_session.boto_session.client('sagemaker')
    )
    print(f"Loaded existing experiment: {experiment_name}")

# Create a Trial for this specific training run
trial_name = f"run-{int(time.time())}"
sagemaker_trial = Trial.create(
    experiment_name=experiment_name,
    trial_name=trial_name,
    sagemaker_boto_client=sagemaker_session.boto_session.client('sagemaker')
)
print(f"Created new trial: {trial_name}")

## Assemble and Run pipeline

In [ ]:
# Create the pipeline by chaining the steps
pipeline = Pipeline(
    name="HuggingFaceFineTuneAndRegisterPipeline",
    parameters=[], # No parameters needed for this basic pipeline
    steps=[step_process, step_train, step_register_model],
    sagemaker_session=sess,
)

# Create or update the pipeline definition in SageMaker
print("Upserting the pipeline definition...")
pipeline.upsert(role_arn=role)

# Start the pipeline execution
print("Starting pipeline execution...")
execution = pipeline.start()

# You can describe the execution to see its status
execution.describe()

## Deploy the model

In [ ]:
# This code would be run in a separate notebook after the pipeline is complete and the model is approved.

# import boto3
# import sagemaker

# region = boto3.Session().region_name
# sm_client = boto3.client("sagemaker", region_name=region)
# role = sagemaker.get_execution_role()

# model_package_group_name = "HuggingFace-Finetune-OPT-Pipeline"

# # Get the latest approved model package
# response = sm_client.list_model_packages(
#     ModelPackageGroupName=model_package_group_name,
#     ModelApprovalStatus="Approved",
#     SortBy="CreationTime",
#     SortOrder="Descending",
#     MaxResults=1
# )

# if not response["ModelPackageSummaryList"]:
#     print("No approved model packages found.")
# else:
#     latest_model_package_arn = response["ModelPackageSummaryList"][0]["ModelPackageArn"]
#     print(f"Latest approved model package: {latest_model_package_arn}")
    
#     # Deploy the model
#     model = sagemaker.ModelPackage(
#         role=role,
#         model_package_arn=latest_model_package_arn,
#         sagemaker_session=sagemaker.Session()
#     )
    
#     endpoint_name = "finetuned-opt-endpoint"
#     model.deploy(
#         initial_instance_count=1,
#         instance_type="ml.g5.2xlarge",
#         endpoint_name=endpoint_name
#     )
    
#     # predictor = sagemaker.predictor.Predictor(endpoint_name=endpoint_name)
#     # # Now you can use the predictor to get inferences